# Oscilaciones inerciales

**Alejandro Jaramillo Moreno**  
Instituto de Ciencias de la Atmósfera y Cambio Climático  
Universidad Nacional Autónoma de México

Las oscilaciones inerciales corresponden al movimiento de una partícula en la atmósfera bajo la influencia exclusiva de la fuerza de Coriolis, en ausencia de fuerzas de presión y fricción.

Este tipo de movimiento surge al considerar las ecuaciones horizontales simplificadas:

$$
\frac{du}{dt} = f v
$$

$$
\frac{dv}{dt} = -f u
$$

donde:

$$
f = 2\Omega \sin\phi
$$

es el parámetro de Coriolis.

El sistema completo incluye también la evolución de la posición:

$$
\frac{dx}{dt} = u, \qquad \frac{dy}{dt} = v
$$

Por lo tanto, se tiene un sistema acoplado para:

- posición: $(x, y)$  
- velocidad: $(u, v)$  

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.integrate import solve_ivp
from IPython.display import HTML

# Velocidad angular de rotación de la Tierra [rad/s]
Omega = 7.292e-5               
# radio de la Tierra [m]
Re = 6371e3               

In [2]:
# latitud inicial [grados]
lat0_deg = 19.0          
# longitud inicial [grados]
lon0_deg = -99    

# velocidad inicial [m/s]
V = 10.0           


# número de pasos de integración
# este valor es para hacer la animación mas rápida
# pero al costo de un menor número de pasos
n_pasos = 100

In [3]:
lat0 = np.deg2rad(lat0_deg)
lon0 = np.deg2rad(lon0_deg)

# parámetro de Coriolis [1/s]
f = 2 * Omega * np.sin(lat0)   

# posición inicial en el plano local [m]
x0, y0 = 0.0, 0.0    

# Periodo inercial
# Mostramos en clase que este movimiento produce una oscilación con
# periodo t=2Pi/f. Usaremos esto para calcular una oscilación completa 
# para construir una animación
T = 2*np.pi/abs(f)   # [s]

# Integramos un periodo completo
t_eval = np.linspace(0, T, n_pasos)

# Radio teórico
R = V / abs(f)

print(f"Latitud inicial = {lat0_deg:.2f}°")
print(f"Longitud inicial = {lon0_deg:.2f}°")
print(f"f = {f:.6e} s^-1")
print(f"Periodo inercial = {T/3600:.2f} h")
print(f"Radio inercial = {R/1000:.2f} km")

Latitud inicial = 19.00°
Longitud inicial = -99.00°
f = 4.748086e-05 s^-1
Periodo inercial = 36.76 h
Radio inercial = 210.61 km


In [4]:
def inertial_system(t, Y):
    x, y, u, v = Y

    dxdt = u
    dydt = v
    dudt = f * v
    dvdt = -f * u

    return [dxdt, dydt, dudt, dvdt]

# Condiciones iniciales
Y0 = [x0, y0, V, 0.0]

# Solución numérica
sol = solve_ivp(inertial_system, [0, T], Y0, t_eval=t_eval)

print(sol)

  message: The solver successfully reached the end of the integration interval.
  success: True
   status: 0
        t: [ 0.000e+00  1.337e+03 ...  1.310e+05  1.323e+05]
        y: [[ 0.000e+00  1.336e+04 ... -1.299e+04  3.678e+02]
            [ 0.000e+00 -4.240e+02 ... -4.094e+02 -1.117e+01]
            [ 1.000e+01  9.980e+00 ...  9.981e+00  9.999e+00]
            [ 0.000e+00 -6.342e-01 ...  6.170e-01 -1.746e-02]]
      sol: None
 t_events: None
 y_events: None
     nfev: 92
     njev: 0
      nlu: 0


In [5]:
x = sol.y[0]
y = sol.y[1]   
u = sol.y[2]
v = sol.y[3]
t = sol.t


# Conversión de desplazamientos locales a lat/lon
lat_traj = lat0+y/Re
lon_traj = lon0+x/(Re*np.cos(lat0))

lat_traj_deg = np.rad2deg(lat_traj)
lon_traj_deg = np.rad2deg(lon_traj)

# Centro teórico del círculo en coordenadas locales
yc_center = y0-V / f
xc_center = x0

# Convertir también el centro a lat/lon
lat_center = lat0+yc_center/Re
lon_center = lon0+xc_center/(Re*np.cos(lat0))

lat_center_deg = np.rad2deg(lat_center)
lon_center_deg = np.rad2deg(lon_center)

In [8]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.set_xlabel("Longitud (°)")
ax.set_ylabel("Latitud (°)")
ax.set_title("Oscilación inercial (solución numérica)")
ax.grid()

# Para que la figura tenga márgenes agradables
lon_margin = 0.1 * (lon_traj_deg.max() - lon_traj_deg.min())
lat_margin = 0.1 * (lat_traj_deg.max() - lat_traj_deg.min())

if lon_margin == 0:
    lon_margin = 0.01
if lat_margin == 0:
    lat_margin = 0.01

ax.set_xlim(lon_traj_deg.min() - lon_margin, lon_traj_deg.max() + lon_margin)
ax.set_ylim(lat_traj_deg.min() - lat_margin, lat_traj_deg.max() + lat_margin)

# Trayectoria completa tenue
ax.plot(lon_traj_deg, lat_traj_deg, '--', alpha=0.3, label="Trayectoria")

# Punto inicial
ax.plot(lon0_deg, lat0_deg, 'go', ms=6, label="Punto inicial")

# Centro teórico
ax.plot(lon_center_deg, lat_center_deg, 'ko', ms=4, label="Centro teórico")

# Elementos animados
line, = ax.plot([], [], lw=2, label="Recorrido")
point, = ax.plot([], [], 'ro', ms=8)
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")


def init():
    line.set_data([], [])
    point.set_data([], [])
    time_text.set_text("")
    return line, point, time_text

def update(frame):
    line.set_data(lon_traj_deg[:frame+1], lat_traj_deg[:frame+1])
    point.set_data([lon_traj_deg[frame]], [lat_traj_deg[frame]])
    time_text.set_text(f"t = {t[frame]/3600:.2f} h")
    return line, point, time_text

anim = FuncAnimation(
    fig,
    update,
    frames=len(t),
    init_func=init,
    interval=50,
    blit=True
)

plt.close(fig)

# Mostrar en Jupyter
HTML(anim.to_jshtml())